In [2]:
# =====================================================
# 1. IMPORT LIBRARIES
# =====================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from sklearn.preprocessing import StandardScaler

plt.style.use('dark_background')

In [3]:
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/krupalpatel07/bmw-stock-historical-dataset/bmw.csv


In [5]:
import plotly.io as pio

pio.renderers.default = 'iframe'

In [4]:
# =====================================================
# 2. LOAD DATA
# =====================================================
file_path = "/kaggle/input/datasets/krupalpatel07/bmw-stock-historical-dataset/bmw.csv"
df = pd.read_csv(file_path)

In [6]:
# =====================================================
# 3. PREPROCESSING
# =====================================================
df.columns = [c.lower() for c in df.columns]
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')
df.set_index('date', inplace=True)

In [7]:
# =====================================================
# 4. HEADER (CARBON UI)
# =====================================================
from IPython.display import display, HTML

def carbon_header(text):
    display(HTML(f"""
    <div style="
        background: linear-gradient(135deg, #0f2027, #203a43, #2c5364);
        padding: 18px; border-radius: 14px; margin-top:18px;
        border:1px solid #00e5ff;">
        <h1 style="color:#7df9ff; text-align:center;">{text}</h1>
    </div>
    """))

carbon_header("📊 Price Track")

In [8]:
# =====================================================
# 5. PRICE VISUAL
# =====================================================
fig = px.line(df, y='close', title='BMW Closing Price')
fig.show()

In [9]:
# =====================================================
# 6. VELOCITY (1st DERIVATIVE)
# =====================================================
carbon_header("🏎️ Price Velocity (1st Derivative)")

df['velocity'] = df['close'].diff()

fig = px.line(df, y='velocity', title='Price Velocity')
fig.show()

In [10]:
# =====================================================
# 7. ACCELERATION (2nd DERIVATIVE)
# =====================================================
carbon_header("⚡ Price Acceleration (2nd Derivative)")

df['acceleration'] = df['velocity'].diff()

fig = px.line(df, y='acceleration', title='Price Acceleration')
fig.show()

In [11]:
# =====================================================
# 8. MOMENTUM CURVATURE
# =====================================================
carbon_header("🌀 Momentum Curvature")

# curvature proxy: acceleration / (1 + velocity^2)^(3/2)
df['curvature'] = df['acceleration'] / (1 + df['velocity']**2)**1.5

fig = px.line(df, y='curvature', title='Curvature')
fig.show()


In [12]:
# =====================================================
# 9. NORMALIZED STATE SPACE
# =====================================================
carbon_header("🧠 State Space (Velocity vs Acceleration)")

state = df[['velocity','acceleration']].dropna()
scaler = StandardScaler()
state_scaled = scaler.fit_transform(state)

state_df = pd.DataFrame(state_scaled, columns=['vel','acc'], index=state.index)

fig = px.scatter(state_df, x='vel', y='acc')
fig.show()

In [13]:
# =====================================================
# 10. REGIME DETECTION (QUADRANT LOGIC)
# =====================================================
carbon_header("⚡ Motion Regimes")

cond_up_acc = (df['velocity'] > 0) & (df['acceleration'] > 0)
cond_up_dec = (df['velocity'] > 0) & (df['acceleration'] < 0)
cond_down_acc = (df['velocity'] < 0) & (df['acceleration'] < 0)
cond_down_dec = (df['velocity'] < 0) & (df['acceleration'] > 0)

regime = np.select(
    [cond_up_acc, cond_up_dec, cond_down_acc, cond_down_dec],
    ['Uptrend Accel','Uptrend Slow','Downtrend Accel','Downtrend Bounce'],
    default='Neutral'
)

df['regime'] = regime

fig = px.scatter(df, x=df.index, y='close', color='regime')
fig.show()

In [14]:
# =====================================================
# 11. SIGNAL ENGINE
# =====================================================
carbon_header("🎯 Velocity Strategy")

signal = (df['velocity'] > 0) & (df['acceleration'] > 0)
df['signal'] = signal.astype(int)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['close'], name='Price'))
fig.add_trace(go.Scatter(x=df.index[df['signal']==1], y=df['close'][df['signal']==1],
                         mode='markers', name='Buy'))
fig.show()

In [15]:
# =====================================================
# 12. EXTREME MOTION DETECTOR
# =====================================================
carbon_header("🔥 Extreme Motion Zones")

z = (df['velocity'] - df['velocity'].mean())/df['velocity'].std()
df['extreme'] = np.abs(z) > 2

fig = px.scatter(df, x=df.index, y='close', color='extreme')
fig.show()

In [16]:
# =====================================================
# 13. FINAL INSIGHTS
# =====================================================
carbon_header("📌 Motion-Based Alpha")

print("""
1. Velocity captures direction strength.
2. Acceleration captures trend change speed.
3. Curvature reveals turning points.
4. Quadrant regimes define market states.
5. Combining velocity + acceleration creates timing edge.
""")

# =====================================================
# END
# =====================================================



1. Velocity captures direction strength.
2. Acceleration captures trend change speed.
3. Curvature reveals turning points.
4. Quadrant regimes define market states.
5. Combining velocity + acceleration creates timing edge.

